In [1]:
from weather_engine.database import engine
import pandas as pd
import xgboost as xgb
from weather_engine.llocv import load_fold, temporal_split_fold


FEATURES = ['rain', 'ws', 'stdwd', 'td', 'rh', 'tdmax', 'tdmin', 'u_vec', 'v_vec']

station_neighbors: pd.DataFrame = pd.read_sql("SELECT * FROM station_neighbors", engine)

In [2]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

all_X = []
all_y = []

all_data = pd.read_sql("SELECT * FROM clean_station_data", engine)
all_data['timestamp'] = pd.to_datetime(all_data['timestamp'])
all_data = all_data.set_index('timestamp').sort_index()
station_frames = {sid: grp.drop(columns='station_id') 
                  for sid, grp in all_data.groupby('station_id')}

for _, row in station_neighbors.iterrows():
    X, y = load_fold(
        int(row["station_id"]),
        int(row["neighbor_1_id"]),
        int(row["neighbor_2_id"]), 
        int(row["neighbor_3_id"]),
        station_frames=station_frames
    )
    all_X.append(X)
    all_y.append(y)
print("All folds loaded.")


all_X = pd.concat(all_X).sort_index()
all_y = pd.concat(all_y).sort_index()

X_train, X_val, y_train, y_val = temporal_split_fold(all_X, all_y)

models = {}
errors = {}

for feature in FEATURES:
    print(f"[{FEATURES.index(feature) + 1}/{len(FEATURES)}] Training RFSI for '{feature}'...", end=' ')
    model = xgb.XGBRegressor(n_jobs=-1)
    model.fit(X_train, y_train[feature])
    preds = model.predict(X_val)
    mae = mean_absolute_error(y_val[feature], preds)
    rmse = root_mean_squared_error(y_val[feature], preds)
    errors[feature] = {'mae': mae, 'rmse': rmse}
    models[feature] = model
    print(f"MAE={mae:.4f}  RMSE={rmse:.4f}")



All folds loaded.
[1/9] Training RFSI for 'rain'... MAE=5.6107  RMSE=203.9911
[2/9] Training RFSI for 'ws'... MAE=0.9812  RMSE=1.3166
[3/9] Training RFSI for 'stdwd'... MAE=6.2437  RMSE=34.4657
[4/9] Training RFSI for 'td'... MAE=1.4058  RMSE=1.8816
[5/9] Training RFSI for 'rh'... MAE=5.8358  RMSE=8.3472
[6/9] Training RFSI for 'tdmax'... MAE=1.4191  RMSE=1.8976
[7/9] Training RFSI for 'tdmin'... MAE=1.4106  RMSE=1.8889
[8/9] Training RFSI for 'u_vec'... MAE=1.1089  RMSE=1.5226
[9/9] Training RFSI for 'v_vec'... MAE=0.9672  RMSE=1.3033
